# 07. Frozen-model external evaluation — separate ElasticNet and XGBoost runs

This paper-ready notebook applies the **prespecified** CCLE-trained models to all 20 external cohorts without external model selection or refitting.

The two multigene families are deliberately executed in **separate cells**:

1. prepare/reuse the shared disk-backed validation workspace;
2. run or resume **ElasticNet only**, with a live `tqdm` protein progress bar;
3. run or resume **XGBoost only**, with an independent progress bar;
4. finalize the portable cohort checkpoints and compact metrics only after both inventories are complete.

The same memory-safe design is retained: the external matrices and prediction matrices remain disk-backed, one HDF5 model is loaded at a time, predictions are generated in bounded sample batches, and no long sample-by-protein table is materialized. Progress is checkpointed to `state.json`, so interrupting either model family does not discard completed proteins.

In [6]:
from pathlib import Path
import gc
import json
import os
import sys

# Bound BLAS/OpenMP/XGBoost threading before importing numerical libraries.
for _thread_var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"
):
    os.environ.setdefault(_thread_var, "1")

# Find paper_validation.py beside the notebook/package or through an explicit override.
search_roots = []
if os.environ.get("PAPER_VALIDATION_MODULE_DIR"):
    search_roots.append(Path(os.environ["PAPER_VALIDATION_MODULE_DIR"]).expanduser().resolve())
cwd = Path.cwd().resolve()
for base in [cwd, *cwd.parents]:
    for candidate in [base, base / "paper_ready_external_validation", base / "notebooks", base / "notebooks2"]:
        if candidate not in search_roots:
            search_roots.append(candidate)
for candidate in search_roots:
    if (candidate / "paper_validation.py").exists():
        sys.path.insert(0, str(candidate))
        MODULE_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "paper_validation.py was not found. Keep the updated module beside this notebook "
        "or set PAPER_VALIDATION_MODULE_DIR."
    )

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
import paper_validation as pv

PROJECT_ROOT = pv.detect_project_root()
RUN_ROOT = Path(os.environ.get(
    "PAPER_VALIDATION_RUN_ROOT",
    PROJECT_ROOT / "results_external_validation_paper_v1",
)).expanduser().resolve()
LAYOUT = pv.make_output_layout(RUN_ROOT)
MANIFEST_PATH = RUN_ROOT / "validation_manifest.json"
SMOKE_TEST = os.environ.get("PAPER_VALIDATION_SMOKE_TEST", "0") == "1"

print("Notebook 07 — separate model-family execution")
print("Shared module:", MODULE_DIR / "paper_validation.py")
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Smoke-test mode:", SMOKE_TEST)

Notebook 07 — separate model-family execution
Shared module: C:\Users\hp\Desktop\new_pRNA\notebooks2\paper_validation.py
Project root: C:\Users\hp\Desktop\new_pRNA
Run root: C:\Users\hp\Desktop\new_pRNA\results_external_validation_paper_v1
Smoke-test mode: False


## 1. Load and validate the frozen manifest

Notebook 06 must have completed successfully. This cell verifies the frozen reference, proxy parameters, and exact model inventory before any external prediction is made.

In [7]:
manifest = pv.load_manifest(MANIFEST_PATH)
MANIFEST_HASH = manifest["manifest_hash"]

if manifest["primary_analysis"]["external_model_selection"] is not False:
    raise RuntimeError("The frozen manifest permits external model selection, which is not paper-ready.")
if manifest["primary_analysis"]["external_sctranslator"] != "removed_incomplete":
    raise RuntimeError("External scTranslator must be complete or explicitly removed.")

for label in ["ccle_reference", "linear_proxy_parameters", "model_inventory"]:
    spec = manifest["files"][label]
    path = Path(spec["path"])
    if not path.exists() or pv.file_sha256(path) != spec["sha256"]:
        raise RuntimeError(f"Frozen contract file is missing or changed: {label} -> {path}")

for cohort, spec in manifest["files"]["cohort_contracts"].items():
    path = Path(spec["path"])
    if not path.exists() or pv.file_sha256(path) != spec["sha256"]:
        raise RuntimeError(f"Cohort contract is missing or changed: {cohort} -> {path}")

reference = joblib.load(manifest["files"]["ccle_reference"]["path"])
proxy_parameters = pv.read_table(manifest["files"]["linear_proxy_parameters"]["path"])
model_inventory = pv.read_table(manifest["files"]["model_inventory"]["path"])

inventory_hash = pv.stable_hash(
    model_inventory.sort_values(["display_model", "protein"]).to_dict("records")
)
if inventory_hash != manifest["model_inventory_hash"]:
    raise RuntimeError("The loaded primary model inventory does not match the manifest hash.")

inventory_summary = (
    model_inventory.groupby("display_model", as_index=False)
    .agg(proteins=("protein", "nunique"), files=("path", "size"))
)
print("Manifest hash:", MANIFEST_HASH)
display(inventory_summary)

Manifest hash: 7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082aed6fc66190da13781e


,display_model,proteins,files
0,ElasticNet,5113,5113
1,XGBoost,5113,5113


## 2. Execution configuration

`RESET_ALL=True` removes the shared evaluation workspace and invalidates previously finalized cohort checkpoints before the ElasticNet cell starts. Leave it `False` to resume.

The progress bar advances **one protein model at a time**. Checkpoints are also flushed periodically, so the bar can be interrupted and resumed without starting from protein 1.

In [8]:
RESET_ALL = os.environ.get("PAPER_VALIDATION_FORCE", "0") == "1"
COMPUTE_SPEARMAN = os.environ.get("PAPER_VALIDATION_SPEARMAN", "1") == "1"
COMPRESSED_NPZ = os.environ.get("PAPER_VALIDATION_COMPRESS_NPZ", "1") == "1"
SAMPLE_BATCH_SIZE = int(os.environ.get(
    "PAPER_VALIDATION_SAMPLE_BATCH", "128" if SMOKE_TEST else "512"
))
CHECKPOINT_EVERY = int(os.environ.get(
    "PAPER_VALIDATION_CHECKPOINT_EVERY", "2" if SMOKE_TEST else "50"
))
# Keep the workspace after finalization by default so either family can be inspected/resumed.
CLEANUP_WORK_AFTER_FINALIZE = os.environ.get("PAPER_VALIDATION_CLEANUP_WORK", "0") == "1"

cohort_bundle_paths = {
    cohort: spec["path"]
    for cohort, spec in manifest["files"]["cohort_contracts"].items()
}
work_dir = LAYOUT["evaluation"] / f"work_{MANIFEST_HASH[:16]}"
state_path = work_dir / "state.json"

def show_family_state():
    if not state_path.exists():
        return pd.DataFrame({
            "display_model": ["ElasticNet", "XGBoost"],
            "completed_proteins": [0, 0],
            "total_proteins": [
                int((model_inventory["display_model"] == "ElasticNet").sum()),
                int((model_inventory["display_model"] == "XGBoost").sum()),
            ],
        })
    state = json.loads(state_path.read_text(encoding="utf-8"))
    rows = []
    for family in ["ElasticNet", "XGBoost"]:
        total = int((model_inventory["display_model"] == family).sum())
        done = int(state.get("progress", {}).get(family, 0))
        rows.append({
            "display_model": family,
            "completed_proteins": done,
            "total_proteins": total,
            "percent": 100.0 * done / total if total else np.nan,
        })
    return pd.DataFrame(rows)

print("RESET_ALL:", RESET_ALL)
print("Sample batch size:", SAMPLE_BATCH_SIZE)
print("Checkpoint every proteins:", CHECKPOINT_EVERY)
print("Work directory:", work_dir)
display(show_family_state())

RESET_ALL: False
Sample batch size: 512
Checkpoint every proteins: 50
Work directory: C:\Users\hp\Desktop\new_pRNA\results_external_validation_paper_v1\evaluation\work_7d25c28c33478194


,display_model,completed_proteins,total_proteins
0,ElasticNet,0,5113
1,XGBoost,0,5113


## 3. Run / resume ElasticNet only

Run this cell independently. It evaluates only the prespecified z-score, \(k=5000\) ElasticNet inventory. XGBoost is not evaluated in this cell.

In [12]:
enet_progress = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=True,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=RESET_ALL,
    cleanup_work_on_success=False,
    model_families=("ElasticNet",),
    show_progress=True,
    finalize=False,
)
RESET_ALL = False  # Never erase the completed ENet run when the XGB cell is executed next.
gc.collect()
display(enet_progress)
display(show_family_state())

ElasticNet proteins:   1%|          | 50/5113 [00:41<1:47:46,  1.28s/protein]

ElasticNet: 50/5,113 proteins


ElasticNet proteins:   2%|▏         | 100/5113 [01:22<52:40,  1.59protein/s] 

ElasticNet: 100/5,113 proteins


ElasticNet proteins:   3%|▎         | 150/5113 [01:53<1:02:39,  1.32protein/s]

ElasticNet: 150/5,113 proteins


... (198 outputs left out)

ElasticNet proteins: 100%|██████████| 5113/5113 [48:18<00:00,  1.76protein/s]

ElasticNet: 5,113/5,113 proteins


,display_model,completed_proteins,total_proteins,fraction_complete,complete,manifest_hash,work_dir
0,ElasticNet,5113,5113,1.0,True,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...
1,XGBoost,5113,5113,1.0,True,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...


,display_model,completed_proteins,total_proteins,percent
0,ElasticNet,5113,5113,100.0
1,XGBoost,5113,5113,100.0


## 4. Run / resume XGBoost only

This cell uses the same disk-backed RNA inputs and workspace but evaluates **only XGBoost**. It can be executed later or resumed independently after interruption.

In [9]:
xgb_progress = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=True,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=False,
    cleanup_work_on_success=False,
    model_families=("XGBoost",),
    show_progress=True,
    finalize=False,
)
gc.collect()
display(xgb_progress)
display(show_family_state())

XGBoost proteins:   1%|          | 50/5113 [01:42<2:58:23,  2.11s/protein]

XGBoost: 50/5,113 proteins


XGBoost proteins:   2%|▏         | 100/5113 [03:21<2:50:19,  2.04s/protein]

XGBoost: 100/5,113 proteins


XGBoost proteins:   3%|▎         | 150/5113 [05:01<2:53:08,  2.09s/protein]

XGBoost: 150/5,113 proteins


... (198 outputs left out)

XGBoost proteins: 100%|██████████| 5113/5113 [2:45:00<00:00,  1.94s/protein]

XGBoost: 5,113/5,113 proteins


,display_model,completed_proteins,total_proteins,fraction_complete,complete,manifest_hash,work_dir
0,ElasticNet,0,5113,0.0,False,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...
1,XGBoost,5113,5113,1.0,True,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...


,display_model,completed_proteins,total_proteins,percent
0,ElasticNet,0,5113,0.0
1,XGBoost,5113,5113,100.0


## 5. Finalize the 20 cohort checkpoints

This cell does **not** rerun either model family. It first requires both progress counters to equal their complete model inventories, then writes the per-cohort NPZ prediction checkpoints, compact metric partitions, transformation audits, and completion-status files consumed by notebooks 08–10.

In [13]:
completion = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=True,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=False,
    cleanup_work_on_success=CLEANUP_WORK_AFTER_FINALIZE,
    model_families=(),
    show_progress=False,
    finalize=True,
)
completion_path = LAYOUT["evaluation"] / "evaluation_completion.csv"
completion.to_csv(completion_path, index=False)
display(completion)

C:\Users\hp\Desktop\new_pRNA\notebooks2\paper_validation.py:1167: RuntimeWarning: divide by zero encountered in divide
  (y - y_mean[None, :]) / y_sd[None, :] - (pred - p_mean[None, :]) / p_sd[None, :],
C:\Users\hp\Desktop\new_pRNA\notebooks2\paper_validation.py:1167: RuntimeWarning: invalid value encountered in divide
  (y - y_mean[None, :]) / y_sd[None, :] - (pred - p_mean[None, :]) / p_sd[None, :],
C:\Users\hp\Desktop\new_pRNA\notebooks2\paper_validation.py:1167: RuntimeWarning: divide by zero encountered in divide
  (y - y_mean[None, :]) / y_sd[None, :] - (pred - p_mean[None, :]) / p_sd[None, :],
C:\Users\hp\Desktop\new_pRNA\notebooks2\paper_validation.py:1167: RuntimeWarning: invalid value encountered in divide
  (y - y_mean[None, :]) / y_sd[None, :] - (pred - p_mean[None, :]) / p_sd[None, :],
C:\Users\hp\Desktop\new_pRNA\notebooks2\paper_validation.py:1167: RuntimeWarning: divide by zero encountered in divide
  (y - y_mean[None, :]) / y_sd[None, :] - (pred - p_mean[None, :]) / p_

,cohort,external_sctranslator,manifest_hash,metric_path,metric_sha256,model_failures,models,normalizer_signature,normalizers_verified,pipeline_version,platform,prediction_path,prediction_sha256,proteins_in_reference,samples,scenarios,status
0,BRAIN,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,df0e72a536f5852d8e823f4b9bc89e09594c31f92b7736...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,LabelFree,C:\Users\hp\Desktop\new_pRNA\results_external_...,546e2bd4076dc63fa9e5cb76cfd0bef5d982a679bcbc57...,5113,71,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
1,COLON,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,8f7552e354c1f010614d7af747e0d010e7979a0c668bd1...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,LabelFree,C:\Users\hp\Desktop\new_pRNA\results_external_...,7e1748877630143958cbf6e662fd1c7dba14faa2b1929e...,5113,86,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
2,LIVER,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,a36faa1db57c0788ed19355b99db92d4a3c967c96f4181...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,LabelFree,C:\Users\hp\Desktop\new_pRNA\results_external_...,a6584039c51e2058537366f2ee082ab7b0c02d027a76d1...,5113,62,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
3,LUNG,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,751c0c4a8aad39385381f6783ffc86ee9e30adf6def62a...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,LabelFree,C:\Users\hp\Desktop\new_pRNA\results_external_...,babf42876b11264de4d53b603697842732de75fabbe154...,5113,76,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
4,PROSTATE,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,14c50030630af49c59b30d4a2ae3188ca96912efb89f5c...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,LabelFree,C:\Users\hp\Desktop\new_pRNA\results_external_...,65f718fd6f9b8149c48b028c0ffa1ba9ab55e269c00a7d...,5113,65,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
5,BRAIN_2021,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,7bb4805f8b9518ac20c20b5f5c61d5436dea250f77dc6f...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,TMT,C:\Users\hp\Desktop\new_pRNA\results_external_...,1fd6f6e138ab87e59a7da1948ba881a6d6c40ab34526ac...,5113,108,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
6,BREAST2020,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,ebba3019304b7c0ba9fd21975b5d23768a79b29bb5660c...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,TMT,C:\Users\hp\Desktop\new_pRNA\results_external_...,f44f2351d6b7ac3acc8c0959eb5a6e8d0d5430953e1149...,5113,122,"[primary_raw, rna_clipped, rna_scale_aligned, ...",complete
7,COLON_2019,removed_incomplete,7d25c28c3347819470e4ee4bd0f00af195f0ba9b00082a...,C:\Users\hp\Desktop\new_pRNA\results_external_...,d8a6331f6d80925a851f387b4434a16b7d1f5aca607578...,0,"[LinearProxy, ElasticNet, XGBoost]",6a7c3a5720b6e1f4ef2e3aa1422c58c0607440033a0811...,10226,external_validation_paper_v1.1,TMT,C:\Users\hp\Desktop\new_pRNA\results_external_...,0fa49f6d8b877487d632f

## 6. Final completeness and provenance audit

In [14]:
completion = pv.validate_evaluation_completion(
    LAYOUT["evaluation_status"],
    MANIFEST_HASH,
    pv.COHORT_TABLE["cohort"].tolist(),
)
if len(completion) != 20:
    raise RuntimeError(f"Expected 20 completed cohorts, found {len(completion)}")
if completion["model_failures"].sum() != 0:
    raise RuntimeError("One or more primary model predictions failed.")

expected_models = {"LinearProxy", "ElasticNet", "XGBoost"}
for models in completion["models"].astype(str):
    # Stored JSON lists may render as strings in the completion DataFrame.
    if not all(name in models for name in expected_models):
        raise RuntimeError(f"Incomplete finalized model set: {models}")

provenance = pv.notebook_provenance(
    "07_evaluation_paper_ready.ipynb",
    MANIFEST_HASH,
    {
        "manifest": str(MANIFEST_PATH.resolve()),
        "model_inventory_hash": manifest["model_inventory_hash"],
        "execution_mode": "separate_enet_xgb_progress_bars",
    },
    {
        "completion_table": str(completion_path.resolve()),
        "prediction_directory": str(LAYOUT["prediction_checkpoints"].resolve()),
        "metric_directory": str(LAYOUT["metric_partitions"].resolve()),
    },
)
pv.atomic_json_dump(provenance, LAYOUT["evaluation"] / "notebook07_provenance.json")

print("Completed cohorts:", len(completion))
print("ElasticNet run separately:", True)
print("XGBoost run separately:", True)
print("Progress bars:", "tqdm protein-wise")
print("Prediction checkpoints use long rows:", False)
print("External scTranslator:", "removed")

Completed cohorts: 20
ElasticNet run separately: True
XGBoost run separately: True
Progress bars: tqdm protein-wise
Prediction checkpoints use long rows: False
External scTranslator: removed
